# Easy Colab Trainer
Paste `BASE_URL` and `BUNDLE_ID`, then run all cells.

In [ ]:
!pip -q install transformers datasets peft accelerate bitsandbytes pyyaml requests

In [ ]:
import requests
from pathlib import Path

BASE_URL = input('BASE_URL (https://...): ').strip().rstrip('/')
BUNDLE_ID = input('BUNDLE_ID (bun_...): ').strip()

if not BASE_URL.startswith('http'):
    raise ValueError('BASE_URL must start with http:// or https://')
if not BUNDLE_ID:
    raise ValueError('BUNDLE_ID is required')

launch = requests.get(f'{BASE_URL}/v1/colab/launch', params={'bundle_id': BUNDLE_ID}, timeout=60)
launch.raise_for_status()
launch_data = launch.json()
BUNDLE_URL = launch_data['env']['BUNDLE_URL']
BASE_MODEL = launch_data['env'].get('BASE_MODEL', 'Qwen/Qwen2.5-3B-Instruct')

script = requests.get(f'{BASE_URL}/v1/colab/train-script', timeout=60)
script.raise_for_status()
Path('train_lora.py').write_text(script.text, encoding='utf-8')

print('Fetched launch config and training script')
print('BASE_MODEL:', BASE_MODEL)
print('BUNDLE_URL:', BUNDLE_URL[:120] + '...')

In [ ]:
!python train_lora.py --bundle-url "$BUNDLE_URL" --base-model "$BASE_MODEL" --output-dir outputs

In [ ]:
from pathlib import Path
metrics_path = Path('outputs/metrics.json')
if metrics_path.exists():
    print(metrics_path.read_text())
else:
    print('metrics.json not found')

!zip -r adapter.zip outputs/adapter

In [ ]:
from google.colab import files
files.download('adapter.zip')